In [1]:
import numpy as np
import pandas as pd
import os
import dotenv
import psycopg
from sqlalchemy import create_engine

import dash
from dash import dcc
from dash import html
from dash.dependencies import Input, Output

import plotly.express as px
import plotly.figure_factory as ff

In [2]:
dotenv.load_dotenv()
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')

dbms = 'postgresql'
package = 'psycopg'
user = 'postgres'
password = POSTGRES_PASSWORD
host = 'localhost'
port = '5432'
db = 'contrans'

engine = create_engine(f'{dbms}+{package}://{user}:{password}@{host}:{port}/{db}')

In [3]:
external_stylesheets = ['https://codepen.io/chriddyp/pen/bWLwgP.css']

# Create features that we need, but won't change depending on what the user does
myquery = '''
SELECT *
FROM members
'''
data = pd.read_sql_query(myquery, con=engine)

In [4]:
data2 = data[['full_name', 'state_abbrev', 'district_code', 'party']]
data2['party'] = data2['party'].str[0]
display_name = [n + ' (' + p + ', ' + s + '-' + str(d) + ')'
                for n, s, d, p in 
                zip(data2['full_name'], 
                    data2['state_abbrev'],
                    data2['district_code'],
                    data2['party'])]
display_name = [x.replace('-0', '') for x in display_name]
display_name

['Robert B. Aderholt (R, AL-4)',
 'Jake Auchincloss (D, MA-4)',
 'Mark E. Amodei (R, NV-2)',
 'Alma S. Adams (D, NC-12)',
 'Pete Aguilar (D, CA-33)',
 'Rick W. Allen (R, GA-12)',
 'Jodey C. Arrington (R, TX-19)',
 'Mark Alford (R, MO-4)',
 'Gabe Amo (D, RI-1)',
 'Yassamin Ansari (D, AZ-3)',
 'Angela D. Alsobrooks (D, MD)',
 'Sanford D. Bishop, Jr. (D, GA-2)',
 'Cliff Bentz (R, OR-2)',
 'Stephanie I. Bice (R, OK-5)',
 'Lauren Boebert (R, CO-4)',
 'Tammy Baldwin (D, WI)',
 'John Boozman (R, AR)',
 'Marsha Blackburn (R, TN)',
 'Gus M. Bilirakis (R, FL-12)',
 'Vern Buchanan (R, FL-16)',
 'John Barrasso (R, WY)',
 'Michael F. Bennet (D, CO)',
 'Richard Blumenthal (D, CT)',
 'Suzanne Bonamici (D, OR-1)',
 'Joyce Beatty (D, OH-3)',
 'Andy Barr (R, KY-6)',
 'Julia Brownley (D, CA-26)',
 'Ami Bera (D, CA-6)',
 'Cory A. Booker (D, NJ)',
 'Brian Babin (R, TX-36)',
 'Donald S. Beyer, Jr. (D, VA-8)',
 'Mike Bost (R, IL-12)',
 'Brendan F. Boyle (D, PA-2)',
 'Don Bacon (R, NE-2)',
 'Jim Banks (R, IN)

In [5]:
dropdown_options = [{'label': y, 'value': x} for x, y in zip(data['bioguide_id'], display_name)]


In [6]:
dropdown_options

[{'label': 'Robert B. Aderholt (R, AL-4)', 'value': 'A000055'},
 {'label': 'Jake Auchincloss (D, MA-4)', 'value': 'A000148'},
 {'label': 'Mark E. Amodei (R, NV-2)', 'value': 'A000369'},
 {'label': 'Alma S. Adams (D, NC-12)', 'value': 'A000370'},
 {'label': 'Pete Aguilar (D, CA-33)', 'value': 'A000371'},
 {'label': 'Rick W. Allen (R, GA-12)', 'value': 'A000372'},
 {'label': 'Jodey C. Arrington (R, TX-19)', 'value': 'A000375'},
 {'label': 'Mark Alford (R, MO-4)', 'value': 'A000379'},
 {'label': 'Gabe Amo (D, RI-1)', 'value': 'A000380'},
 {'label': 'Yassamin Ansari (D, AZ-3)', 'value': 'A000381'},
 {'label': 'Angela D. Alsobrooks (D, MD)', 'value': 'A000382'},
 {'label': 'Sanford D. Bishop, Jr. (D, GA-2)', 'value': 'B000490'},
 {'label': 'Cliff Bentz (R, OR-2)', 'value': 'B000668'},
 {'label': 'Stephanie I. Bice (R, OK-5)', 'value': 'B000740'},
 {'label': 'Lauren Boebert (R, CO-4)', 'value': 'B000825'},
 {'label': 'Tammy Baldwin (D, WI)', 'value': 'B001230'},
 {'label': 'John Boozman (R, 

In [7]:
b = 'A000055'

myquery = f'''
SELECT *
FROM members
WHERE bioguide_id = '{b}'
'''
member_info = pd.read_sql_query(myquery, con=engine)
member_info = member_info.drop(['bioguide_id', 'image', 'fec_id', 'bioname', 'icpsr'],
    axis=1)
ff.create_table(member_info.T.reset_index().rename({'index':'',0:''}, axis=1))

In [9]:
myquery = f'''
SELECT image
FROM members
WHERE bioguide_id = '{b}'
'''
pd.read_sql_query(myquery, con=engine)['image'][0]

'https://www.congress.gov/img/member/a000055_200.jpg'

In [16]:
myquery = f'''
SELECT c.comparison_member,
    c.agree,
    m.left_right_ideology,
    m.party
FROM members m
INNER JOIN (
    SELECT vc.comparison_member,
        vc.agree
    FROM members m
    INNER JOIN vote_compare vc
        ON m.bioname = vc.bioname
    WHERE m.bioguide_id = '{b}'
) c
    ON m.bioname = c.comparison_member

'''
pd.read_sql_query(myquery, con=engine)

,comparison_member,agree,left_right_ideology,party
0,"ROGERS, Harold Dallas (Hal)",0.957295,0.333,Republican
1,"SMITH, Christopher Henry",0.925267,0.182,Republican
2,"HOYER, Steny Hamilton",0.330961,-0.381,Democrat
3,"KAPTUR, Marcia Carolyn (Marcy)",0.405694,-0.343,Democrat
4,"MFUME, Kweisi",0.313167,-0.440,Democrat
...,...,...,...,...
437,"ANSARI, Yassamin",0.327402,-0.506,Democrat
438,"BARRETT, Tom",0.921708,0.509,Republican
439,"BAUMGARTNER, Michael",0.935943,0.489,Republican
440,"BEGICH, Nicholas J., III",0.896797,0.634,Republican


In [21]:
myquery = f'''



SELECT c.comparison_member,
    c.agree,
    m.left_right_ideology,
    m.party
FROM members m
INNER JOIN (SELECT vc.comparison_member, vc.agree
FROM members m
INNER JOIN vote_compare vc
    ON m.bioname = vc.bioname
WHERE bioguide_id = '{b}') c
    ON m.bioname = c.comparison_member


'''

vote_data = pd.read_sql_query(myquery, con=engine)

In [22]:
import plotly.express as px

In [27]:
fig = px.scatter(vote_data,
                 x = 'left_right_ideology',
                 y = 'agree',
                 hover_name = 'comparison_member',
                 color = 'party',
                 color_discrete_map={'Democrat': 'blue','Republican': 'red'})
fig.show()

# Color mapping for parties

